In [ ]:
from collections.abc import Sequence
from functools import partial
from pathlib import Path

import geopandas as gpd
import polars as pl
import traitlets
from lonboard import Map
from lonboard.models import ViewState

from h3xplorer import core

In [ ]:
data_dir = Path("D:/programming/data/h3xplorer")
census2011_dir = data_dir / "census2011"
census2021_dir = data_dir / "census2021"

In [ ]:
c2011_df = pl.read_csv(census2011_dir / "ks404ew_car_or_van_availability_clean.csv")
c2021_df = pl.read_csv(census2021_dir / "rs008_car_or_van_availability_clean.csv")
display(c2011_df, c2021_df)

In [ ]:
c2011_df = c2011_df.with_columns(
    (
        pl.col("All categories: Car or van availability") - pl.col("No cars or vans in household")
    ).alias("1 or more cars or vans in household")
)

In [ ]:
display(c2011_df.sum(), c2021_df.sum())

In [ ]:
c2011_points = gpd.read_file(census2011_dir / "lsoa_centroids_pop_weighted.gpkg")
c2021_points = gpd.read_file(census2021_dir / "lsoa_centroids_pop_weighted.gpkg")
display(c2011_points, c2021_points)

In [ ]:
def _get_points(gdf: gpd.GeoDataFrame, retain_cols: list[str]) -> pl.DataFrame:
    return pl.DataFrame(
        {"x": gdf.geometry.x, "y": gdf.geometry.y} | {col: gdf.loc[:, col] for col in retain_cols}
    )


c2011_xys = _get_points(c2011_points, ["lsoa11cd"])
c2021_xys = _get_points(c2021_points, ["LSOA21CD"])
display(c2011_xys, c2021_xys)

In [ ]:
c2011_data = (
    c2011_df.join(c2011_xys, left_on="mnemonic", right_on="lsoa11cd")
    .select(
        "x",
        "y",
        no_car="No cars or vans in household",
        has_car="1 or more cars or vans in household",
        total="All categories: Car or van availability",
    )
    .with_columns((pl.col("has_car") / pl.col("total")).alias("percent_with_car"))
)
c2021_data = (
    c2021_df.join(c2021_xys, left_on="mnemonic", right_on="LSOA21CD")
    .select(
        "x",
        "y",
        no_car="No cars or vans in household",
        has_car="1 or more cars or vans in household",
        total="Total",
    )
    .with_columns((pl.col("has_car") / pl.col("total")).alias("percent_with_car"))
)
display(c2011_data, c2021_data)

In [ ]:
hex_size = 6
map_2011_totalhh = core.xy_plot(c2011_data, "x", "y", 27700, hex_size, "total", "sum")
map_2021_totalhh = core.xy_plot(c2021_data, "x", "y", 27700, hex_size, "total", "sum")

In [ ]:
map_2011_percentcar_mean = core.xy_plot(
    c2011_data, "x", "y", 27700, hex_size, "percent_with_car", "mean"
)
map_2021_percentcar_mean = core.xy_plot(
    c2021_data, "x", "y", 27700, hex_size, "percent_with_car", "mean"
)

In [ ]:
def _link_maps(event: traitlets.utils.bunch.Bunch, other_maps: Sequence[Map] = ()) -> None:
    if isinstance(event.get("new"), ViewState):
        for lonboard_map in other_maps:
            lonboard_map.view_state = event["new"]

In [ ]:
map_2011_totalhh.observe(partial(_link_maps, other_maps=[map_2021_totalhh]))
map_2021_totalhh.observe(partial(_link_maps, other_maps=[map_2011_totalhh]))
display(map_2011_totalhh, map_2021_totalhh)

In [ ]:
map_2011_percentcar_mean.observe(partial(_link_maps, other_maps=[map_2021_percentcar_mean]))
map_2021_percentcar_mean.observe(partial(_link_maps, other_maps=[map_2011_percentcar_mean]))
display(map_2011_percentcar_mean, map_2021_percentcar_mean)